## 1. Configuration

In [ ]:
import os, sys, time, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
warnings.filterwarnings('ignore')
PROJECT_DIR = os.getcwd()
OUTPUT_DIR  = f'{PROJECT_DIR}/output'

os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_STATE = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

FORCE_EXTRACT_MIMIC = False
FORCE_EXTRACT_EICU  = False
SEQUENCE_LEN   = 48
N_COVARIATES   = 23
ADD_MISS_IND   = False

D_MODEL, N_HEADS, N_LAYERS, D_FF = 128, 4, 4, 256
NOISE_DIM, HIDDEN_DIM, DROPOUT   = 8, 128, 0.1

FORCE_RETRAIN  = False
EPOCHS_STAGE1  = 100
PATIENCE       = 10
LEARNING_RATE  = 1e-4
WEIGHT_DECAY   = 1e-5
BATCH_SIZE     = 128
MIN_EPOCHS_STAGE1 = 0
SELECT_ON_C       = False
USE_ADVERSARIAL   = False

LAMBDA_ADV, LAMBDA_SURV, LAMBDA_VFD = 0.0, 2.5, 2.0
LAMBDA_CONSIST, LAMBDA_GATE         = 0.3, 0.05
LAMBDA_IPM, LAMBDA_DR, LAMBDA_GP    = 0.0, 1.0, 10.0
LAMBDA_FACT, LAMBDA_ANCHOR          = 8.0, 0.1
LAMBDA_JOINT = 0.5
PROP_CLIP    = 0.1
USE_OVERLAP_WEIGHTS = True

K_FOLDS          = 5
USE_HYBRID_BASE  = True
BASE_LEARNER     = 't-learner'
STACK_N_ITER, STACK_N_BAGS, STACK_BAG_FRAC = 12, 8, 0.5
VFD_HORIZON_DAYS = 28

APPLY_EICU_PARITY  = True
EICU_EXCLUDE_CRASH = False

np.random.seed(RANDOM_STATE); torch.manual_seed(RANDOM_STATE)
print(f'device={DEVICE}  seed={RANDOM_STATE}')

## 2. Imports

In [ ]:
from matplotlib.patches import Patch, FancyBboxPatch
from scipy import stats
import numpy as np, torch, time, os, importlib
import types, torch.nn as nn

from models.dinirs import DINIRSModel
from models.baselines import run_baselines, CustomCausalSurvivalForest
from models.baselines import CustomCausalSurvivalForest as _CSF
from models.baselines import cross_fitted_tree_base
import models.ensemble as fmod
from training.train import CensoringAwareAdversarialLoss
from utils.extraction import create_dataloaders, NIRSTwinDataset
from utils.generalization import cross_fitted_ite, subgroup_robustness
import utils.generalization as _gen
from utils.extraction import (
    init_client, run_mimic_extraction, normalize_and_mask,
    propensity_score_match_baseline, FEATURE_COLS,
    TEMPORAL_FEATURE_COLS,
    identify_eicu_arf, apply_eicu_exclusions, extract_eicu_cohort,
    assign_eicu_treatment, extract_eicu_covariates, compute_eicu_vfd28,
    extract_eicu_temporal, build_eicu_temporal_sequences)
from utils.metrics import (
    match_on_predicted_benefit, c_for_benefit, c_for_benefit_ci, paired_c_for_benefit_test,
    calibration_for_benefit, risk_stratified_benefit, cross_fitted_propensity, aipw_policy_value,
    policy_value_table, vfd_days_gained, benefit_harm_interaction, benefit_calibration_slope,
    standardized_mean_differences, plot_ite_distribution)
import utils.metrics as _met
from training.train import train_stage1, train_stage2, evaluate, DEFAULT_CONFIG
import utils.mice as _mice
importlib.reload(_mice)
from utils.mice import mice_pmm, rubin_pool

print('imports done and  torch version is:', torch.__version__)

In [ ]:
def _code_consts(fn):
    seen, out, stack = set(), set(), [fn.__code__]
    while stack:
        c = stack.pop()
        if id(c) in seen:
            continue
        seen.add(id(c))
        for k in c.co_consts:
            if hasattr(k, 'co_consts'):
                stack.append(k)
            else:
                out.add(k)
    return out

_checks = {
    'metrics.py':
        hasattr(_met, 'match_on_predicted_benefit'),
    'metrics.py':
        200_000 in _code_consts(_met.c_for_benefit_ci),
    'baselines.py':
        any(isinstance(s, str) and 'IPCW DISABLED' in s
            for s in _code_consts(_CSF.fit)),
    'generalization.py':
        1000 in _code_consts(_gen.cross_fitted_ite),
}
for _name, _ok in _checks.items():
    print(f"  [{'PASS' if _ok else 'FAIL'}] {_name}")

if not all(_checks.values()):
    raise RuntimeError(
        "STALE MODULES IN MEMORY - the kernel is executing code older than the "
        "files on disk. Fix: Kernel > Restart Kernel and Run All Cells. "
        "Do NOT trust any number produced while this gate fails.")



## 3. MIMIC-IV cohort

In [ ]:
MIMIC_CACHE = f'{OUTPUT_DIR}/mimic_cache'; os.makedirs(MIMIC_CACHE, exist_ok=True)
_ts, _co = f'{MIMIC_CACHE}/mimic_temporal.npz', f'{MIMIC_CACHE}/mimic_cohort.csv'

if not FORCE_EXTRACT_MIMIC and os.path.exists(_ts):
    print('Loading cached MIMIC-IV...')
    _c = np.load(_ts)
    X_all, W_all, VFD_all, D_all = _c['X'], _c['W'], _c['VFD'], _c['D']
    valid_ids = _c['valid_ids']
    df_cohort = pd.read_csv(_co)
    df_vfd    = pd.read_csv(f'{MIMIC_CACHE}/mimic_vfd.csv')
    df_cov    = pd.read_csv(f'{MIMIC_CACHE}/mimic_cov.csv')
    _rawp = f'{MIMIC_CACHE}/mimic_cov_raw.csv'
    df_cov_raw = pd.read_csv(_rawp) if os.path.exists(_rawp) else None
else:
    print('Running BigQuery extraction...')
    data = run_mimic_extraction(init_client(), seq_len=SEQUENCE_LEN)
    X_all, W_all = data['X'], data['W']
    VFD_all, D_all, valid_ids = data['VFD'], data['D'], data['valid_ids']
    df_cohort, df_vfd, df_cov = data['df_cohort'], data['df_vfd'], data['df_cov']
    np.savez_compressed(_ts, X=X_all, W=W_all, VFD=VFD_all, D=D_all,
                        valid_ids=valid_ids)
    df_cohort.to_csv(_co, index=False)
    df_vfd.to_csv(f'{MIMIC_CACHE}/mimic_vfd.csv', index=False)
    df_cov.to_csv(f'{MIMIC_CACHE}/mimic_cov.csv', index=False)
    df_cov_raw = data.get('df_cov_raw')
    if df_cov_raw is not None:
        df_cov_raw.to_csv(f'{MIMIC_CACHE}/mimic_cov_raw.csv', index=False)
        print('  cached raw (pre-imputation) covariates for the MICE cell')

X_psm, W_psm, VFD_psm, D_psm = X_all, W_all, VFD_all, D_all
print(f'  X={X_all.shape}  N={len(W_all):,}  '
      f'NIRS={int((W_all==1).sum()):,}  IMV={int((W_all==0).sum()):,}')

## 4. Covariate

In [ ]:
_cov  = df_cov.drop_duplicates('stay_id').set_index('stay_id')
_ids  = np.asarray(valid_ids)
_have = np.array([i in _cov.index for i in _ids])
X_base = np.zeros((len(_ids), len(FEATURE_COLS)))
X_base[_have] = _cov.loc[_ids[_have], FEATURE_COLS].astype(float).fillna(0.0).values

_psm_local, ps_model, ps = propensity_score_match_baseline(
    X_base[_have], W_all[_have], FEATURE_COLS,
    caliper_scale=0.1, random_state=RANDOM_STATE)
matched_idx = np.where(_have)[0][_psm_local]
matched_mask = np.zeros(len(_ids), dtype=bool); matched_mask[matched_idx] = True
print(f'Evaluation subset: {matched_mask.sum():,} patients\n')
standardized_mean_differences(X_base[matched_idx], W_all[matched_idx],
                              feature_names=FEATURE_COLS)

## 5. Scale, split, and the cross-fitted tree base

In [ ]:
X_scaled, pad_masks, ts_scaler = normalize_and_mask(
    X_psm, N_COVARIATES, add_missing_indicators=ADD_MISS_IND, robust=True)

X_model = X_scaled
N_COV_MODEL = X_model.shape[-1]
print(f'model input channels: {N_COV_MODEL}')

N = len(X_model)
_rng = np.random.RandomState(RANDOM_STATE)
_m, _u = np.where(matched_mask)[0], np.where(~matched_mask)[0]
_rng.shuffle(_m); _rng.shuffle(_u)
n_te = min(int(N * 0.10), len(_m)); n_va = int(N * 0.10)
test_idx = _m[:n_te]
_rest = np.concatenate([_m[n_te:], _u]); _rng.shuffle(_rest)
val_idx, train_idx = _rest[:n_va], _rest[n_va:]
print(f'split: train={len(train_idx)} val={len(val_idx)} test={len(test_idx)} '
      f'(test 100% matched)')

X_tab = X_scaled[:, :, :N_COVARIATES].mean(axis=1)
_b = cross_fitted_tree_base(X_tab, W_psm, VFD_psm, K=K_FOLDS,
                            seed=RANDOM_STATE, learner=BASE_LEARNER)
tau_base = np.nan_to_num(_b['tau_base'], nan=0.0).astype(np.float32)
mu0_base, mu1_base = _b['mu0'], _b['mu1']
print(f"tree base: c-for-benefit (all N) = "
      f"{c_for_benefit(tau_base, VFD_psm, W_psm)['c_for_benefit']:.3f}")

## 6. Model and training configuration

In [ ]:

loss_fn = CensoringAwareAdversarialLoss(
    lambda_adv=LAMBDA_ADV, lambda_surv=LAMBDA_SURV, lambda_vfd=LAMBDA_VFD,
    lambda_consist=LAMBDA_CONSIST, lambda_gate=LAMBDA_GATE,
    lambda_ipm=LAMBDA_IPM, lambda_dr=LAMBDA_DR)

config = DEFAULT_CONFIG.copy()
config.update({
    'n_covariates': N_COV_MODEL, 'd_model': D_MODEL, 'n_heads': N_HEADS,
    'n_layers': N_LAYERS, 'd_ff': D_FF, 'noise_dim': NOISE_DIM,
    'hidden_dim': HIDDEN_DIM, 'dropout': DROPOUT,
    'epochs_stage1': EPOCHS_STAGE1, 'patience': PATIENCE,
    'lr_generator': LEARNING_RATE, 'lr_discriminator': LEARNING_RATE,
    'lr_encoder': LEARNING_RATE, 'weight_decay': WEIGHT_DECAY,
    'batch_size': BATCH_SIZE, 'device': str(DEVICE), 'save_dir': OUTPUT_DIR,
    'min_epochs_stage1': MIN_EPOCHS_STAGE1, 'select_on_c_for_benefit': SELECT_ON_C,
    'use_adversarial': USE_ADVERSARIAL, 'lambda_adv': LAMBDA_ADV,
    'lambda_vfd': LAMBDA_VFD, 'lambda_consist': LAMBDA_CONSIST,
    'lambda_gate': LAMBDA_GATE, 'lambda_fact': LAMBDA_FACT,
    'lambda_anchor': LAMBDA_ANCHOR, 'prop_clip': PROP_CLIP,
    'lambda_ipm': LAMBDA_IPM, 'lambda_dr': LAMBDA_DR, 'lambda_gp': LAMBDA_GP,
    'lambda_joint': LAMBDA_JOINT, 'use_overlap_weights': USE_OVERLAP_WEIGHTS,
})
_probe = DINIRSModel(
    n_covariates=N_COV_MODEL, d_model=D_MODEL, n_heads=N_HEADS,
    n_layers=N_LAYERS, d_ff=D_FF, noise_dim=NOISE_DIM,
    hidden_dim=HIDDEN_DIM, dropout=DROPOUT)
print(f'parameters: {sum(p.numel() for p in _probe.parameters()):,}')
del _probe

## 7. Cross-fitted out-of-fold ITE

In [ ]:
_TRAJ_CH = [3, 4, 5, 6, 8, 9, 10, 11, 12, 13, 14, 19]

def _traj_cov_summary(self, x, pad_mask=None):
    x = x[..., :self.n_cov_raw]
    B, Tt, C = x.shape
    if pad_mask is not None:
        v = (~pad_mask).float().unsqueeze(-1)
    else:
        v = torch.ones(B, Tt, 1, device=x.device, dtype=x.dtype)
    denom = v.sum(1).clamp(min=1.0)
    mean = (x * v).sum(1) / denom
    t = torch.arange(Tt, device=x.device, dtype=x.dtype).view(1, Tt, 1)
    tbar = (t * v).sum(1) / denom
    xc = x - mean.unsqueeze(1)
    tc = t - tbar.unsqueeze(1)
    slope = (tc * xc * v).sum(1) / (tc * tc * v).sum(1).clamp(min=1e-6)
    sd = ((xc * xc * v).sum(1) / denom).clamp(min=0.0).sqrt()
    last = x[:, -1, :]
    fidx = v.squeeze(-1).argmax(dim=1)
    first = x[torch.arange(B, device=x.device), fidx, :]
    delta = last - first
    sel = torch.as_tensor(_TRAJ_CH, device=x.device, dtype=torch.long)
    extra = torch.cat([slope.index_select(1, sel), sd.index_select(1, sel),
                       last.index_select(1, sel), delta.index_select(1, sel)], dim=-1)
    return torch.cat([mean, extra], dim=-1)

_N_FUSION_EXTRA = 4 * len(_TRAJ_CH)

def _build():
    m = DINIRSModel(
        n_covariates=N_COV_MODEL, d_model=D_MODEL, n_heads=N_HEADS,
        n_layers=N_LAYERS, d_ff=D_FF, noise_dim=NOISE_DIM,
        hidden_dim=HIDDEN_DIM, dropout=DROPOUT).to(DEVICE)
    new_cov_dim = m.n_cov_raw + _N_FUSION_EXTRA
    m.predictor.cov_dim = new_cov_dim
    m.predictor.cov_proj = nn.Sequential(
        nn.Linear(new_cov_dim, HIDDEN_DIM), nn.GELU(), nn.Dropout(DROPOUT)
    ).to(DEVICE)
    m.cov_summary = types.MethodType(_traj_cov_summary, m)
    return m

_OOF = f'{OUTPUT_DIR}/tau_oof_twin.npy'
if FORCE_RETRAIN or not os.path.exists(_OOF):
    t0 = time.time()
    _cf = cross_fitted_ite(_build, train_stage1, train_stage2, loss_fn, config,
                           X_model, W_psm, VFD_psm, D_psm, pad_masks,
                           K=K_FOLDS, seed=RANDOM_STATE,
                           save_dir=f'{OUTPUT_DIR}/_crossfit_final',
                           tau_base=tau_base)
    tau_twin = _cf['ite_oof']
    np.save(_OOF, tau_twin)
    print(f'cross-fit finished in {(time.time()-t0)/60:.1f} min')
else:
    tau_twin = np.load(_OOF); print(f'loaded cached OOF twin from {_OOF}')

_c = c_for_benefit_ci(tau_twin, VFD_psm, W_psm)
print(f"out-of-fold: c-for-benefit = {_c['point']:.3f} "
      f"[{_c['lower']:.3f}, {_c['upper']:.3f}]")

## 8. Baselines Model

In [ ]:
_BL = f'{OUTPUT_DIR}/baselines_oof.npz'
if FORCE_RETRAIN or not os.path.exists(_BL):
    _cf_cf  = cross_fitted_tree_base(X_tab, W_psm, VFD_psm, K=K_FOLDS,
                                     seed=RANDOM_STATE, learner='causal-forest')
    tau_cf  = np.asarray(_cf_cf['tau_base'], dtype=float)
    _csf = CustomCausalSurvivalForest(n_trees=100, horizon=float(VFD_HORIZON_DAYS),
                                  random_state=RANDOM_STATE)
    _csf.fit(X_tab, W_psm, VFD_psm, D_psm)
    tau_csf = np.asarray(_csf.predict_ite(X_tab), dtype=float).ravel()
    np.savez(_BL, tau_cf=tau_cf, tau_csf=tau_csf)
else:
    _z = np.load(_BL); tau_cf, tau_csf = _z['tau_cf'], _z['tau_csf']

BASELINES = {'T-Learner': np.asarray(tau_base, dtype=float),
             'Causal Forest': tau_cf,
             'Causal Survival Forest': tau_csf}
for _n, _v in BASELINES.items():
    _ci = c_for_benefit_ci(_v, VFD_psm, W_psm)
    print(f"  {_n:<24} c = {_ci['point']:.3f} "
          f"[{_ci['lower']:.3f}, {_ci['upper']:.3f}]")

## 9. Final estimator: Cross-validated ensemble

In [ ]:
LIB = dict(BASELINES); LIB['DINIRS twin'] = np.asarray(tau_twin, dtype=float)
TWIN_KEYS = {'DINIRS twin'}

r_base, r_full = fmod.incremental_value(
    LIB, TWIN_KEYS, VFD_psm, W_psm, k_folds=K_FOLDS, seed=RANDOM_STATE,
    n_iter=STACK_N_ITER, verbose=True, bagged=True, n_bags=STACK_N_BAGS,
    bag_frac=STACK_BAG_FRAC, guard_to_best=True)

tau_final, mean_w = fmod.rescale_to_days(r_full['tau_z'], LIB, r_full['weights'])
tau_basestack, _  = fmod.rescale_to_days(r_base['tau_z'], LIB, r_base['weights'])
np.save(f'{OUTPUT_DIR}/tau_final.npy', tau_final)

print(f"\nmean weights: { {k: round(v,3) for k,v in mean_w.items() if v>0} }")
print(f"selection-c per fold: {[round(s,4) for s in r_full['sel_c']]}")
print(f"realized pooled c   : "
      f"{c_for_benefit(tau_final, VFD_psm, W_psm)['c_for_benefit']:.4f}")
print(f"ITE: sd={tau_final.std():.2f}  ATE={tau_final.mean():+.2f}  "
      f"%benefit={100*(tau_final>0).mean():.1f}%")

In [ ]:
_rng = np.random.RandomState(RANDOM_STATE)
_order = _rng.permutation(len(tau_final))
_fold = np.empty(len(tau_final), dtype=int)
for _i, _ix in enumerate(_order):
    _fold[_ix] = _i % K_FOLDS

tau_cal = np.empty_like(np.asarray(tau_final, dtype=float))
_slopes, _ints = [], []
for _k in range(K_FOLDS):
    _tr = np.where(_fold != _k)[0]
    _te = np.where(_fold == _k)[0]
    _A, _B = match_on_predicted_benefit(tau_final[_tr], W_psm[_tr])
    _a, _b = _tr[_A], _tr[_B]
    _obs = VFD_psm[_a] - VFD_psm[_b]
    _pred = 0.5 * (tau_final[_a] + tau_final[_b])
    _v = np.var(_pred)
    _sl = float(np.cov(_pred, _obs)[0, 1] / _v) if _v > 1e-12 else 1.0
    _ic = float(_obs.mean() - _sl * _pred.mean())
    if _sl <= 0:
        _sl, _ic = 1.0, float(_obs.mean() - _pred.mean())
    _slopes.append(_sl); _ints.append(_ic)
    tau_cal[_te] = _ic + _sl * tau_final[_te]

print(f"per-fold recalibration slope     : {np.round(_slopes, 3).tolist()}")

_n_clip = int((np.abs(tau_cal) > 28.0).sum())
tau_cal = np.clip(tau_cal, -28.0, 28.0)
print(f"clipped to [-28, +28]: {_n_clip} of {len(tau_cal)} patients affected")

print(f"per-fold recalibration intercept : {np.round(_ints, 3).tolist()}")

_e = e_hat if 'e_hat' in globals() else cross_fitted_propensity(
    X_tab, W_psm, k_folds=K_FOLDS, seed=RANDOM_STATE, clip=PROP_CLIP)
print(f"\n{'':<22}{'ATE':>8}{'% NIRS':>9}{'c-for-benefit':>16}{'E-stat':>9}{'policy value':>14}")
for _lab, _v in [('twin stack', np.asarray(tau_final, dtype=float)),
                 ('stack RECALIBRATED', tau_cal),
                 ('T-Learner', np.asarray(BASELINES['T-Learner'], dtype=float)),
                 ('twin alone', np.asarray(tau_twin, dtype=float))]:
    _ci = c_for_benefit_ci(_v, VFD_psm, W_psm, n_boot=200)
    _cal = calibration_for_benefit(_v, VFD_psm, W_psm, verbose=False)
    _pv = aipw_policy_value((_v > 0).astype(int), W_psm, VFD_psm, _e,
                            mu0_base, mu1_base)
    print(f"  {_lab:<20}{_v.mean():>+8.2f}{100*(_v>0).mean():>8.1f}%"
          f"{_ci['point']:>10.3f}      {_cal.get('e_stat_for_benefit', np.nan):>7.2f}"
          f"{(_pv['value'] if isinstance(_pv, dict) else float(_pv)):>14.3f}")

_flip = 100 * float(((np.asarray(tau_final) > 0) != (tau_cal > 0)).mean())
print("\nREAD. Recalibration preserves the ranking WITHIN each fold exactly")
print("(the map is affine with positive slope).")

np.save(f'{OUTPUT_DIR}/tau_final_recalibrated.npy', tau_cal)
print(f"\nsaved -> {OUTPUT_DIR}/tau_final_recalibrated.npy")

tau_final_uncalibrated = np.asarray(tau_final, dtype=float).copy()
np.save(f'{OUTPUT_DIR}/tau_final_uncalibrated.npy', tau_final_uncalibrated)
tau_final = tau_cal
np.save(f'{OUTPUT_DIR}/tau_final.npy', tau_final)
print("\ntau_final now holds the RECALIBRATED estimator")

In [ ]:
ALL = {'DINIRS (final stack)': tau_final,
       'twin alone': np.asarray(tau_twin, dtype=float),
       'baseline-only stack': tau_basestack, **BASELINES}

print(f"{'estimator':<26}{'c-for-benefit':>16}{'E-stat (days)':>16}")
RESULTS = {}
for _n, _v in ALL.items():
    _ci = c_for_benefit_ci(_v, VFD_psm, W_psm)
    _cal = calibration_for_benefit(_v, VFD_psm, W_psm, verbose=False)
    _e = _cal.get('e_stat_for_benefit', float('nan')) if isinstance(_cal, dict) else float('nan')
    RESULTS[_n] = {'c': _ci, 'e_stat': _e}
    print(f"  {_n:<24}{_ci['point']:>7.3f} "
          f"[{_ci['lower']:.3f},{_ci['upper']:.3f}]{_e:>14.2f}")

print('\npaired tests')
for _n in ['T-Learner', 'Causal Forest', 'Causal Survival Forest']:
    print(f'\nvs {_n}:')
    _pt = paired_c_for_benefit_test(tau_final, BASELINES[_n], VFD_psm, W_psm,
                                    name_a='DINIRS (final)', name_b=_n)
    display(pd.Series(_pt))
print('\nINCREMENTAL VALUE OF THE TWIN')
_pt = paired_c_for_benefit_test(tau_final, tau_basestack, VFD_psm, W_psm,
                                name_a='full stack', name_b='baseline-only stack')
display(pd.Series(_pt))

## 10. Clinical impact
Doubly-robust policy value, VFD days gained, benefit/harm interaction.

In [ ]:
e_hat = cross_fitted_propensity(X_tab, W_psm, k_folds=K_FOLDS,
                                seed=RANDOM_STATE, clip=PROP_CLIP)

print('\nDoubly-robust policy value (VFD-28 days)')
PV = policy_value_table({k: v for k, v in ALL.items()
                         if k in ('DINIRS (final stack)', 'T-Learner',
                                'Causal Forest', 'Causal Survival Forest')},
                        W_psm, VFD_psm, e_hat, mu0_base, mu1_base)
display(pd.DataFrame(PV).T.drop(columns=['psi'], errors='ignore'))

print('\nVentilator-free days gained vs observed practice')
GAIN = vfd_days_gained(tau_final, W_psm, VFD_psm, e_hat, mu0_base, mu1_base,
                       n_boot=2000, seed=RANDOM_STATE)
display(pd.Series(GAIN))

print('\nPredicted BENEFIT vs predicted HARM (the Wissel test)')
INTER = benefit_harm_interaction(tau_final, W_psm, VFD_psm,
                                 label='DINIRS (final)')
display(pd.Series(INTER))
for _n in ('T-Learner',):
    display(pd.Series(benefit_harm_interaction(BASELINES[_n], W_psm, VFD_psm, label=_n)))

## 11 Multiple imputation with Rubin's Rules

In [ ]:
M_IMPUTATIONS = 50
MICE_CYCLES   = 5

_bl_src = df_cov_raw if ('df_cov_raw' in globals()
                         and df_cov_raw is not None) else df_cov
if _bl_src is df_cov:
    print('  !! mimic_cov_raw.csv not found: falling back to the PMM-filled')
_bl = (_bl_src.drop_duplicates('stay_id')
              .set_index('stay_id')
              .reindex(np.asarray(valid_ids))[FEATURE_COLS]
              .astype(float).reset_index(drop=True))
_cont_cols = [c for c in _bl.columns
              if _bl[c].dropna().nunique() > 2 and _bl[c].isna().any()]
print(f"columns with missingness to impute: {len(_cont_cols)}")
if len(_cont_cols) == 0:
    print("  *** NOTHING TO IMPUTE. The pooled intervals below are the")

_imps = mice_pmm(_bl, _cont_cols, m=M_IMPUTATIONS, n_iter=MICE_CYCLES, k=5,
                 seed=RANDOM_STATE, outcome=VFD_psm, include_outcome=True)

_pol, _polv, _gain, _gainv, _inter, _interv = [], [], [], [], [], []
_policy = (tau_final > 0).astype(int)
for _i, _d in enumerate(_imps):
    _X = _d[FEATURE_COLS].fillna(_d[FEATURE_COLS].median()).values
    _e = cross_fitted_propensity(_X, W_psm, k_folds=K_FOLDS,
                                 seed=RANDOM_STATE + _i, clip=PROP_CLIP,
                                 verbose=False)
    _m = aipw_policy_value(_policy, W_psm, VFD_psm, _e, mu0_base, mu1_base)
    _o = aipw_policy_value(W_psm,   W_psm, VFD_psm, _e, mu0_base, mu1_base)
    _pol.append(_m['value']);      _polv.append(_m['se'] ** 2)
    _d_psi = _m['psi'] - _o['psi']
    _gain.append(float(_d_psi.mean()))
    _gainv.append(float(_d_psi.var(ddof=1) / len(_d_psi)))
    _r = benefit_harm_interaction(tau_final, W_psm, VFD_psm, verbose=False)
    _inter.append(_r['interaction']); _interv.append(_r['se'] ** 2)

print(f"\nPOOLED ACROSS {M_IMPUTATIONS} IMPUTATIONS (Rubin's Rules)")
print("\nPolicy value of the model-guided rule (VFD-28 days):")
POOL_POLICY = rubin_pool(_pol, _polv, name="policy value")
display(pd.Series(POOL_POLICY))
print("\nVFD-28 days gained vs observed practice (per patient):")
POOL_GAIN = rubin_pool(_gain, _gainv, name="VFD gained")
display(pd.Series(POOL_GAIN))
print(f"  => {100*POOL_GAIN['estimate']:+.1f} VFD per 100 patients "
      f"[{100*POOL_GAIN['lo']:+.1f}, {100*POOL_GAIN['hi']:+.1f}]")
print("\nbenefit-vs-harm interaction (VFD-28 days):")
POOL_INTER = rubin_pool(_inter, _interv, name="interaction")
display(pd.Series(POOL_INTER))
print("\n  FMI is the share of uncertainty coming from imputation rather than")
print("  from the data. Above 0.5, increase M_IMPUTATIONS.")

In [ ]:
df_full = _cov.loc[np.asarray(valid_ids), FEATURE_COLS].reset_index()
df_full['Treatment_W'] = W_psm
df_full['vfd28']       = VFD_psm
df_full['delta']       = D_psm
df_full['ipcw_weight'] = 1.0
df_full['propensity']  = e_hat
df_full['tau_dr']      = tau_final
df_full['tau_twin']    = np.asarray(tau_twin, dtype=float)
df_full['tau_tree_base'] = np.asarray(tau_base, dtype=float)
df_full['tau_tl']      = BASELINES['T-Learner']
df_full['tau_cf']      = BASELINES['Causal Forest']
df_full['tau_csf']     = BASELINES['Causal Survival Forest']
print(f"df_full {df_full.shape}   tau_dr = out-of-fold final stack")
print(f"  ATE={tau_final.mean():+.2f}  sd={tau_final.std():.2f}  "
      f"%benefit={100*(tau_final>0).mean():.1f}%")

## 12. Heterogeneity

In [ ]:
fig = plot_ite_distribution(tau_final, model_name='DINIRS (out-of-fold)',
                           save_path=f'{OUTPUT_DIR}/fig_ite_distribution.png')
plt.show()

_sofa = X_scaled[:, :, 19].mean(axis=1)
_band = np.array(['low', 'mid', 'high'])[np.digitize(_sofa, np.nanpercentile(_sofa, [33, 66]))]
print('\nc-for-benefit within SOFA bands (does it hold across severity?)')
_sg = subgroup_robustness(tau_final, VFD_psm, W_psm, _band, 'SOFA band')
display(pd.DataFrame(_sg).T)

print('\n risk-stratified benefit')
risk_baseline = -np.asarray(mu0_base, dtype=float)
_rs = risk_stratified_benefit(tau_final, VFD_psm, W_psm, risk_baseline,
                              n_strata=3, labels=['low risk', 'mid', 'high risk'])
display(pd.DataFrame(_rs).T)

## 13. Sensitivity to unmeasured confounding

In [ ]:
from utils.metrics import (compute_e_value_for_ate,
                           rosenbaum_sensitivity_bounds, gate_ablation_ci)

_ate = float(tau_final.mean())
print(f'final-model ATE = {_ate:+.3f} VFD-28 days\n')

_ev = compute_e_value_for_ate(_ate, float(VFD_psm.std()),
                              treatment_prevalence=float(W_psm.mean()))
print(f'E-value: {_ev}')

_rb = rosenbaum_sensitivity_bounds(tau_final, W_psm, VFD_psm)
print(f'Rosenbaum bounds: {_rb}')

## 14. eICU external validation

In [ ]:
EICU_CACHE = f'{OUTPUT_DIR}/eicu_cache'; os.makedirs(EICU_CACHE, exist_ok=True)
_ets = f'{EICU_CACHE}/eicu_temporal.npz'
_etx = f'{EICU_CACHE}/eicu_tx.csv'

if not FORCE_EXTRACT_EICU and os.path.exists(_ets) and os.path.exists(_etx):
    print('Loading cached eICU...')
    _c = np.load(_ets)
    X_eicu_ts, W_eicu_ts = _c['X_eicu_ts'], _c['W_eicu_ts']
    VFD_eicu_ts, D_eicu_ts = _c['VFD_eicu_ts'], _c['D_eicu_ts']
    eicu_valid_ids = _c['eicu_valid_ids']
    df_eicu_tx = pd.read_csv(_etx)
else:
    print('Running eICU BigQuery extraction (first run)...')
    df_eicu_tx = assign_eicu_treatment(extract_eicu_cohort())
    if APPLY_EICU_PARITY:
        df_eicu_tx = identify_eicu_arf(df_eicu_tx)
        df_eicu_tx = apply_eicu_exclusions(df_eicu_tx,
                                           exclude_crash=EICU_EXCLUDE_CRASH)
        print(f'  after parity filters: {len(df_eicu_tx):,}')
    df_eicu_cov = extract_eicu_covariates(df_eicu_tx)
    df_eicu_vfd = compute_eicu_vfd28(df_eicu_tx)
    X_eicu_ts, W_eicu_ts, VFD_eicu_ts, D_eicu_ts, eicu_valid_ids = (
        build_eicu_temporal_sequences(extract_eicu_temporal(df_eicu_tx),
                                      df_eicu_tx, df_eicu_vfd, df_eicu_cov,
                                      seq_len=SEQUENCE_LEN))
    np.savez_compressed(_ets, X_eicu_ts=X_eicu_ts, W_eicu_ts=W_eicu_ts,
                        VFD_eicu_ts=VFD_eicu_ts, D_eicu_ts=D_eicu_ts,
                        eicu_valid_ids=eicu_valid_ids)
    df_eicu_tx.to_csv(_etx, index=False)
    df_eicu_cov.to_csv(f'{EICU_CACHE}/eicu_cov.csv', index=False)
    df_eicu_vfd.to_csv(f'{EICU_CACHE}/eicu_vfd.csv', index=False)
    print(f'  cached to {EICU_CACHE}/')

print(f'eICU cohort: {len(W_eicu_ts):,}  '
      f'NIRS {int((W_eicu_ts==1).sum()):,} / IMV {int((W_eicu_ts==0).sum()):,}')

In [ ]:
if True:
    from models.baselines import CustomTLearner
    from utils.generalization import _predict_ite

    X_eicu, W_eicu = X_eicu_ts, W_eicu_ts
    VFD_eicu, D_eicu = VFD_eicu_ts, D_eicu_ts
    print(f'eICU: {X_eicu.shape}  NIRS={int((W_eicu==1).sum()):,} '
          f'IMV={int((W_eicu==0).sum()):,}')
    print(f'  raw arm gap: NIRS {VFD_eicu[W_eicu==1].mean():.2f} vs '
          f'IMV {VFD_eicu[W_eicu==0].mean():.2f} = '
          f'{VFD_eicu[W_eicu==1].mean()-VFD_eicu[W_eicu==0].mean():+.2f} days '
          f'(MIMIC: +3.46) -- the cohorts are NOT on the same scale')

    X_eicu_s, pad_eicu, _ = normalize_and_mask(
        X_eicu, N_COVARIATES, add_missing_indicators=ADD_MISS_IND,
        robust=True, reference_scaler=ts_scaler)
    X_eicu_tab = X_eicu_s[:, :, :N_COVARIATES].mean(axis=1)

    _tl = CustomTLearner(random_state=RANDOM_STATE)
    _tl.fit(X_tab, W_psm, VFD_psm)
    tau_eicu_tree = np.asarray(_tl.predict_ite(X_eicu_tab)[0], dtype=float).ravel()

    _preds = []
    for _k in range(K_FOLDS):
        _ck = f'{OUTPUT_DIR}/_crossfit_final/fold{_k}/best_stage2.pth'
        if not os.path.exists(_ck):
            continue
        _m = _build()
        _m.load_state_dict(torch.load(_ck, map_location=DEVICE))
        _preds.append(_predict_ite(_m, X_eicu_s, W_eicu, VFD_eicu, D_eicu,
                                   pad_eicu, DEVICE, tau_base=tau_eicu_tree))
        del _m
    if _preds:
        tau_eicu_twin = np.mean(_preds, axis=0)
        print(f'  transported twin: ensemble of {len(_preds)} MIMIC fold models')
    else:
        tau_eicu_twin = None
        print('  no fold checkpoints found -- run section 7 first')

    print('\nDISCRIMINATION (rank-based, scale-free)')
    _EI = {'MIMIC tree -> eICU': tau_eicu_tree}
    if tau_eicu_twin is not None:
        _EI['MIMIC twin -> eICU'] = tau_eicu_twin
    for _n, _v in _EI.items():
        _ci = c_for_benefit_ci(_v, VFD_eicu, W_eicu)
        print(f"  {_n:<22} c = {_ci['point']:.3f} "
              f"[{_ci['lower']:.3f}, {_ci['upper']:.3f}]  ATE={_v.mean():+.2f}")
    if tau_eicu_twin is not None:
        paired_c_for_benefit_test(tau_eicu_twin, tau_eicu_tree, VFD_eicu, W_eicu,
                                  name_a='twin', name_b='tree')

    print('\nCALIBRATION: in the large, then after recalibration')
    EICU_CAL = {}
    for _n, _v in _EI.items():
        EICU_CAL[_n] = benefit_calibration_slope(_v, W_eicu, VFD_eicu, label=_n)
    display(pd.DataFrame(EICU_CAL).T)

    print('\nCLINICAL: does the benefit/harm split replicate?')
    _bh = {_n: benefit_harm_interaction(_v, W_eicu, VFD_eicu, label=_n)
           for _n, _v in _EI.items()}
    display(pd.DataFrame(_bh).T)

**Clinical subgroup forest**

In [ ]:

fig, ax = plt.subplots(figsize=(12, 8))
groups = []
for label, lo, hi in [('SOFA 0-3 (Low)', -1, 3), ('SOFA 4-6 (Moderate)', 3, 6),
                        ('SOFA 7-10 (High)', 6, 10), ('SOFA >10 (Very High)', 10, 99)]:
    sub = df_full[(df_full['sofa_X'] > lo) & (df_full['sofa_X'] <= hi)]
    groups.append((label, sub['tau_dr'].mean(), sub['tau_dr'].std()/np.sqrt(len(sub)), len(sub), 'SOFA'))
for label, lo, hi in [('Age 18-40', 17, 40), ('Age 41-60', 40, 60),
                        ('Age 61-75', 60, 75), ('Age >75', 75, 120)]:
    sub = df_full[(df_full['age_X'] > lo) & (df_full['age_X'] <= hi)]
    groups.append((label, sub['tau_dr'].mean(), sub['tau_dr'].std()/np.sqrt(len(sub)), len(sub), 'Age'))
for label, lo, hi in [('P/F <100 (Severe ARDS)', 0, 100), ('P/F 100-200 (Moderate)', 100, 200),
                        ('P/F 200-300 (Mild)', 200, 300), ('P/F >300 (No ARDS)', 300, 9999)]:
    sub = df_full[(df_full['pf_ratio_X'] > lo) & (df_full['pf_ratio_X'] <= hi)]
    groups.append((label, sub['tau_dr'].mean(), sub['tau_dr'].std()/np.sqrt(len(sub)), len(sub), 'P/F Ratio'))
for col, name in [('copd_X','COPD'),('chf_X','CHF'),('sepsis_X','Sepsis'),('immunosuppressed_X','Immunosuppressed')]:
    for val, suffix in [(1, ' (+)'), (0, ' (-)')]:
        sub = df_full[df_full[col]==val]
        groups.append((name+suffix, sub['tau_dr'].mean(), sub['tau_dr'].std()/np.sqrt(len(sub)), len(sub), 'Comorbidity'))

labels = [g[0] for g in groups]
means = [g[1] for g in groups]
ses = [g[2] for g in groups]
ns = [g[3] for g in groups]
cats = [g[4] for g in groups]
colors_map = {'SOFA': '#2196F3', 'Age': '#4CAF50', 'P/F Ratio': '#FF9800', 'Comorbidity': '#9C27B0', 'Overall Cohort': '#E53935'}
colors = [colors_map[c] for c in cats]
y_pos = np.arange(len(groups))

x_min = min(m - 1.96*s for m, s in zip(means, ses))
x_max = max(m + 1.96*s for m, s in zip(means, ses))
x_pad = (x_max - x_min) * 0.15

ax.axvline(x=0, color='red', linestyle='--', linewidth=1.5, alpha=0.7)
ax.barh(y_pos, means, xerr=[1.96*s for s in ses], height=0.6, color=colors,
        alpha=0.7, edgecolor='black', linewidth=0.5, capsize=3)

x_text = x_max + x_pad * 0.3
for i, n in enumerate(ns):
    ax.text(x_text, i, f'n={n:,}', va='center', fontsize=9)

ax.set_xlim(x_min - x_pad, x_max + x_pad * 1.5)
ax.set_yticks(y_pos)
ax.set_yticklabels(labels, fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Estimated ITE (VFD-28 days, positive = NIRS beneficial)', fontsize=11)

prev_cat = None
for i, cat in enumerate(cats):
    if prev_cat and cat != prev_cat:
        ax.axhline(y=i-0.5, linestyle='-', linewidth=0.5, alpha=0.5)
    prev_cat = cat

ax.legend(handles=[Patch(facecolor=v, label=k, alpha=0.7) for k,v in colors_map.items()],
          loc='lower center', fontsize=8, ncol=len(colors_map))
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig_subgroup_forest.tif', dpi=300, bbox_inches='tight')
plt.show()

**Waterfall of ranked ITE**

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
df_sorted = df_full.sort_values('tau_dr')
ites = df_sorted['tau_dr'].values; n = len(ites)
sofa = df_sorted['sofa_X'].values
colors_wf = np.where(sofa > 10, '#E53935', np.where(sofa > 6, '#FF9800', np.where(sofa > 3, '#FFC107', '#4CAF50')))
ax.bar(np.arange(n), ites, width=1.0, color=colors_wf, edgecolor='none', alpha=0.8)
ax.axhline(y=0, color='red', linestyle='--', linewidth=1.5, alpha=0.8)
nirs_pct = (ites > 0).sum() / n * 100
ax.text(n*0.75, max(ites)*0.7, f'NIRS beneficial\n({nirs_pct:.2f}%)', fontsize=10, ha='center', color='#2E7D32', fontweight='bold')
ax.text(n*0.20, min(ites)*0.8, f'IMV beneficial\n({100-nirs_pct:.2f}%)', fontsize=10, ha='center', color='#C62828', fontweight='bold')
ax.set_xlabel('Patients (ranked by ITE)', fontsize=11); ax.set_ylabel('ITE (VFD-28 days)', fontsize=11)
ax.legend(handles=[Patch(facecolor=c, label=l) for c,l in [('#4CAF50','SOFA 0-3'),('#FFC107','SOFA 4-6'),('#FF9800','SOFA 7-10'),('#E53935','SOFA >10')]], 
          loc='center left', fontsize=9)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig_waterfall_ite.tif', dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
sofa_groups = [('0-3', 0, 3), ('4-6', 4, 6), ('7-10', 7, 10), ('>10', 11, 99)]
colors_sofa = ['#4CAF50', '#FFC107', '#FF9800', '#E53935']
recommend_nirs = df_full['tau_dr'] > 0; got_nirs = df_full['Treatment_W'] == 1

sofa_means, sofa_sems, sofa_labels_list = [], [], []
concordance_data = []
for label, lo, hi in sofa_groups:
    sub = df_full[(df_full['sofa_X'] >= lo) & (df_full['sofa_X'] <= hi)]
    sofa_means.append(sub['tau_dr'].mean())
    sofa_sems.append(1.96 * sub['tau_dr'].std() / np.sqrt(len(sub)))
    sofa_labels_list.append(f'SOFA {label}\n(n={len(sub)})')
    rec = recommend_nirs[sub.index]; got = got_nirs[sub.index]
    conc = (rec == got).mean() * 100
    conc_mask = rec == got; disc_mask = ~conc_mask
    vfd_c = sub.loc[conc_mask, 'vfd28'].mean() if conc_mask.sum() > 0 else 0
    vfd_d = sub.loc[disc_mask, 'vfd28'].mean() if disc_mask.sum() > 0 else 0
    concordance_data.append((conc, vfd_c, vfd_d))

axes[0].bar(range(4), sofa_means, yerr=sofa_sems, color=colors_sofa, edgecolor='black', linewidth=0.5, alpha=0.8, capsize=4)
axes[0].axhline(y=0, color='red', linestyle='--', linewidth=1)
axes[0].set_xticks(range(4)); axes[0].set_xticklabels(sofa_labels_list, fontsize=9)
axes[0].set_ylabel('Mean ITE (VFD-28 days)'); axes[0].set_title('(A) ITE by SOFA Severity', fontweight='bold')

conc_rates = [c[0] for c in concordance_data]
axes[1].bar(range(4), conc_rates, color=colors_sofa, edgecolor='black', linewidth=0.5, alpha=0.8)
axes[1].axhline(y=50, color='gray', linestyle=':', linewidth=1)
for i, v in enumerate(conc_rates): axes[1].text(i, v+1, f'{v:.0f}%', ha='center', fontsize=9, fontweight='bold')
axes[1].set_xticks(range(4)); axes[1].set_xticklabels(sofa_labels_list, fontsize=9)
axes[1].set_ylabel('Concordance (%)'); axes[1].set_title('(B) Model-Clinician Concordance', fontweight='bold'); axes[1].set_ylim(0,100)

x = np.arange(4); w = 0.35
axes[2].bar(x-w/2, [c[1] for c in concordance_data], w, color='#2196F3', label='Concordant', edgecolor='black', linewidth=0.5)
axes[2].bar(x+w/2, [c[2] for c in concordance_data], w, color='#FF5722', label='Discordant', edgecolor='black', linewidth=0.5)
axes[2].set_xticks(range(4)); axes[2].set_xticklabels(sofa_labels_list, fontsize=9)
axes[2].set_ylabel('Mean VFD-28 (days)'); axes[2].set_title('(C) Outcomes: Concordant vs Discordant', fontweight='bold'); axes[2].legend(fontsize=9)
plt.suptitle('Treatment Effect Heterogeneity Stratified by Organ Dysfunction Severity', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.savefig(f'{OUTPUT_DIR}/fig_sofa_stratified.png', dpi=200, bbox_inches='tight'); plt.show()

**Cross-method ITE comparison**

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].scatter(df_full['tau_dr'], df_full['tau_tl'], alpha=0.2, s=8, c='#2196F3', edgecolors='none')
axes[0].axhline(0, color='red', linestyle='--', linewidth=0.8, alpha=0.6); axes[0].axvline(0, color='red', linestyle='--', linewidth=0.8, alpha=0.6)
agr1 = ((df_full['tau_dr']>0)==(df_full['tau_tl']>0)).mean()*100
axes[0].set_title(f'(A) DINIRS vs T-Learner\nAgreement: {agr1:.0f}%', fontweight='bold')
axes[0].set_xlabel('DINIRS ITE'); axes[0].set_ylabel('T-Learner ITE')

axes[1].scatter(df_full['tau_dr'], df_full['tau_cf'], alpha=0.2, s=8, c='#FF9800', edgecolors='none')
axes[1].axhline(0, color='red', linestyle='--', linewidth=0.8, alpha=0.6); axes[1].axvline(0, color='red', linestyle='--', linewidth=0.8, alpha=0.6)
agr2 = ((df_full['tau_dr']>0)==(df_full['tau_cf']>0)).mean()*100
axes[1].set_title(f'(B) DINIRS vs Causal Forest\nAgreement: {agr2:.0f}%', fontweight='bold')
axes[1].set_xlabel('DINIRS ITE'); axes[1].set_ylabel('Causal Forest ITE')

sofa_grps = [('0-3',0,3),('4-6',4,6),('7-10',7,10),('>10',11,99)]
x = np.arange(4); w = 0.25
for j,(col,nm,clr) in enumerate([('tau_dr','DINIRS','#2196F3'),('tau_tl','T-Learner','#4CAF50'),('tau_cf','Causal Forest','#FF9800')]):
    pcts = [(df_full[(df_full['sofa_X']>=lo)&(df_full['sofa_X']<=hi)][col]>0).mean()*100 for _,lo,hi in sofa_grps]
    axes[2].bar(x+(j-1)*w, pcts, w, color=clr, label=nm, edgecolor='black', linewidth=0.5, alpha=0.8)
axes[2].axhline(50, color='gray', linestyle=':', linewidth=1)
axes[2].set_xticks(x); axes[2].set_xticklabels([f'SOFA {l}' for l,_,_ in sofa_grps], fontsize=9)
axes[2].set_ylabel('% Recommended NIRS'); axes[2].set_title('(C) NIRS Rate by SOFA Across Methods', fontweight='bold'); axes[2].legend(fontsize=9); axes[2].set_ylim(0,105)
plt.suptitle('Cross-Method ITE Comparison', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.savefig(f'{OUTPUT_DIR}/fig_method_comparison_detailed.png', dpi=200, bbox_inches='tight'); plt.show()

**Clinical decision matrix (SOFA x P/F)**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sofa_bins = [-np.inf,3,6,10,np.inf]; sofa_lab = ['0-3','4-6','7-10','>10']
pf_bins = [-np.inf,100,200,300,np.inf]; pf_lab = ['<100','100-200','200-300','>300']
df_full['sofa_bin'] = pd.cut(df_full['sofa_X'], bins=sofa_bins, labels=sofa_lab, include_lowest=True)
df_full['pf_bin'] = pd.cut(df_full['pf_ratio_X'], bins=pf_bins, labels=pf_lab, include_lowest=True)
piv_ite = df_full.pivot_table(values='tau_dr', index='pf_bin', columns='sofa_bin', aggfunc='mean')
piv_n = df_full.pivot_table(values='tau_dr', index='pf_bin', columns='sofa_bin', aggfunc='count')

ite_abs_max = max(abs(piv_ite.values[~np.isnan(piv_ite.values)].min()), abs(piv_ite.values[~np.isnan(piv_ite.values)].max()))
ite_vbound = max(ite_abs_max * 1.2, 0.5)

im = axes[0].imshow(piv_ite.values, cmap='RdYlGn', aspect='auto', vmin=-ite_vbound, vmax=ite_vbound)
axes[0].set_xticks(range(4)); axes[0].set_xticklabels(sofa_lab)
axes[0].set_yticks(range(4)); axes[0].set_yticklabels(pf_lab)
axes[0].set_xlabel('SOFA Score'); axes[0].set_ylabel('P/F Ratio')
axes[0].set_title('(A) Mean ITE by SOFA x P/F', fontweight='bold')
for i in range(4):
    for j in range(4):
        v = piv_ite.values[i,j]; n = piv_n.values[i,j]
        if not np.isnan(v):
            tc = 'white' if abs(v) > ite_vbound * 0.6 else 'black'
            axes[0].text(j, i, f'{v:+.1f}\nn={int(n)}', ha='center', va='center', fontsize=9, fontweight='bold', color=tc)
plt.colorbar(im, ax=axes[0], label='Mean ITE (VFD-28 days)')

piv_pct = df_full.pivot_table(values='tau_dr', index='pf_bin', columns='sofa_bin', aggfunc=lambda x: (x>0).mean()*100)
im2 = axes[1].imshow(piv_pct.values, cmap='RdYlGn', aspect='auto', vmin=20, vmax=80)
axes[1].set_xticks(range(4)); axes[1].set_xticklabels(sofa_lab)
axes[1].set_yticks(range(4)); axes[1].set_yticklabels(pf_lab)
axes[1].set_xlabel('SOFA Score'); axes[1].set_ylabel('P/F Ratio')
axes[1].set_title('(B) % NIRS Beneficial by SOFA x P/F', fontweight='bold')
for i in range(4):
    for j in range(4):
        v = piv_pct.values[i,j]; n = piv_n.values[i,j]
        if not np.isnan(v): axes[1].text(j, i, f'{v:.0f}%\nn={int(n)}', ha='center', va='center', fontsize=9, fontweight='bold', color='white' if v>70 or v<30 else 'black')
plt.colorbar(im2, ax=axes[1], label='% NIRS Beneficial')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig_clinical_decision_matrix.tif', dpi=300, bbox_inches='tight')
plt.show()


**ROX / P-F stratified**

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

rox_bins = [-np.inf,4,6,8,10,np.inf]; rox_labs = ['<4\n(High fail)','4-6','6-8','8-10','>10\n(Low fail)']
df_full['rox_bin'] = pd.cut(df_full['rox_index_X'], bins=rox_bins, labels=rox_labs, include_lowest=True)
rox_m, rox_se, rox_n = [], [], []
for lab in rox_labs:
    sub = df_full[df_full['rox_bin']==lab]
    rox_m.append(sub['tau_dr'].mean()); rox_se.append(1.96*sub['tau_dr'].std()/np.sqrt(max(len(sub),1))); rox_n.append(len(sub))
clrs_rox = ['#E53935','#FF9800','#FFC107','#8BC34A','#4CAF50']
axes[0].bar(range(5), rox_m, yerr=rox_se, color=clrs_rox, edgecolor='black', linewidth=0.5, alpha=0.8, capsize=4)
axes[0].axhline(0, color='red', linestyle='--', linewidth=1)
axes[0].set_xticks(range(5)); axes[0].set_xticklabels(rox_labs, fontsize=8)
for i,n in enumerate(rox_n): axes[0].text(i, rox_m[i]+rox_se[i]+0.1, f'n={n}', ha='center', fontsize=8, color='gray')
axes[0].set_ylabel('Mean ITE (VFD-28 days)'); axes[0].set_xlabel('ROX Index')
axes[0].set_title('(A) ITE by ROX Index', fontweight='bold')

pf_cuts = [-np.inf,100,150,200,250,300,np.inf]; pf_labs = ['<100','100-150','150-200','200-250','250-300','>300']
df_full['pf_det'] = pd.cut(df_full['pf_ratio_X'], bins=pf_cuts, labels=pf_labs, include_lowest=True)
pf_m, pf_se, pf_n = [], [], []
for lab in pf_labs:
    sub = df_full[df_full['pf_det']==lab]
    pf_m.append(sub['tau_dr'].mean()); pf_se.append(1.96*sub['tau_dr'].std()/np.sqrt(max(len(sub),1))); pf_n.append(len(sub))
clrs_pf = ['#E53935','#FF5722','#FF9800','#FFC107','#8BC34A','#4CAF50']
axes[1].bar(range(6), pf_m, yerr=pf_se, color=clrs_pf, edgecolor='black', linewidth=0.5, alpha=0.8, capsize=4)
axes[1].axhline(0, color='red', linestyle='--', linewidth=1)
axes[1].set_xticks(range(6)); axes[1].set_xticklabels(pf_labs, fontsize=8)
for i,n in enumerate(pf_n): axes[1].text(i, pf_m[i]+pf_se[i]+0.1, f'n={n}', ha='center', fontsize=8, color='gray')
axes[1].set_ylabel('Mean ITE (VFD-28 days)'); axes[1].set_xlabel('P/F Ratio')
axes[1].set_title('(B) ITE by P/F Ratio', fontweight='bold')

nirs_ben = df_full[df_full['tau_dr']>0]; imv_ben = df_full[df_full['tau_dr']<=0]
cvars = ['sofa_X','age_X','pf_ratio_X','rox_index_X','hr_mean_X','rr_mean_X','lactate_X']
cvlabs = ['SOFA','Age','P/F Ratio','ROX Index','Heart Rate','Resp Rate','Lactate']
x_pos = np.arange(len(cvars)); w = 0.35
axes[2].barh(x_pos-w/2, [nirs_ben[v].mean() for v in cvars], w, color='#4CAF50', label='NIRS beneficial', alpha=0.8, edgecolor='black', linewidth=0.5)
axes[2].barh(x_pos+w/2, [imv_ben[v].mean() for v in cvars], w, color='#E53935', label='IMV beneficial', alpha=0.8, edgecolor='black', linewidth=0.5)
axes[2].set_yticks(x_pos); axes[2].set_yticklabels(cvlabs, fontsize=9)
axes[2].set_xlabel('Mean Value'); axes[2].set_title('(C) NIRS vs IMV Beneficial', fontweight='bold'); axes[2].legend(fontsize=9)

plt.suptitle('Respiratory-Specific Treatment Effect Analysis', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.savefig(f'{OUTPUT_DIR}/fig_respiratory_stratified.png', dpi=200, bbox_inches='tight'); plt.show()

**KM curves by predicted ITE group**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

nirs_pts = df_full[df_full['Treatment_W']==1]
high = nirs_pts[nirs_pts['tau_dr']>0]; low = nirs_pts[nirs_pts['tau_dr']<=0]
for grp, data, clr, sty in [(f'NIRS Recommended (n={len(high)})', high, '#4CAF50', '-'),
                              (f'IMV Recommended (n={len(low)})', low, '#E53935', '--')]:
    sv = np.sort(data['vfd28'].values)
    sf = 1 - np.arange(1, len(sv)+1)/len(sv)
    axes[0].step(sv, sf, where='post', color=clr, linestyle=sty, linewidth=2, label=grp)
axes[0].axvline(high['vfd28'].mean(), color='#4CAF50', linestyle=':', alpha=0.5)
axes[0].axvline(low['vfd28'].mean(), color='#E53935', linestyle=':', alpha=0.5)
axes[0].text(high['vfd28'].mean()+0.3, 0.95, f'Mean: {high["vfd28"].mean():.1f}d', fontsize=8, color='#4CAF50')
axes[0].text(low['vfd28'].mean()+0.3, 0.90, f'Mean: {low["vfd28"].mean():.1f}d', fontsize=8, color='#E53935')
axes[0].set_xlabel('VFD-28 (days)'); axes[0].set_ylabel('Fraction >= x')
axes[0].set_title('(A) NIRS Recipients by Predicted Benefit', fontweight='bold'); axes[0].legend(fontsize=9)

imv_pts = df_full[df_full['Treatment_W']==0]
hi_imv = imv_pts[imv_pts['tau_dr']<=0]; lo_imv = imv_pts[imv_pts['tau_dr']>0]
for grp, data, clr, sty in [(f'IMV Recommended (n={len(hi_imv)})', hi_imv, '#2196F3', '-'),
                              (f'NIRS Recommended (n={len(lo_imv)})', lo_imv, '#FF9800', '--')]:
    sv = np.sort(data['vfd28'].values)
    sf = 1 - np.arange(1, len(sv)+1)/len(sv)
    axes[1].step(sv, sf, where='post', color=clr, linestyle=sty, linewidth=2, label=grp)
axes[1].set_xlabel('VFD-28 (days)'); axes[1].set_ylabel('Fraction >= x')
axes[1].set_title('(B) IMV Recipients by Predicted Benefit', fontweight='bold'); axes[1].legend(fontsize=9)

_, p_nirs = stats.mannwhitneyu(high['vfd28'], low['vfd28'], alternative='greater')
_, p_imv = stats.mannwhitneyu(hi_imv['vfd28'], lo_imv['vfd28'], alternative='greater')
plt.suptitle('VFD-28 Survival Curves by Predicted Treatment Benefit', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.savefig(f'{OUTPUT_DIR}/fig_km_ite_stratified.png', dpi=200, bbox_inches='tight'); plt.show()
print(f'NIRS recipients: Mann-Whitney p = {p_nirs:.2e}')
print(f'IMV recipients: Mann-Whitney p = {p_imv:.2e}')

## 15. Export

In [ ]:
df_full.to_csv(f'{OUTPUT_DIR}/model_ite_results_FINAL.csv', index=False)
_out = f'{OUTPUT_DIR}/model_ite_results_FINAL.csv'
df_full.to_csv(_out, index=False)

## 16. Architecture ablation study

In [ ]:
_LOSS_BASE = dict(lambda_adv=LAMBDA_ADV, lambda_surv=LAMBDA_SURV,
                  lambda_vfd=LAMBDA_VFD, lambda_consist=LAMBDA_CONSIST,
                  lambda_gate=LAMBDA_GATE, lambda_ipm=LAMBDA_IPM,
                  lambda_dr=LAMBDA_DR)

ABLATIONS = {
    'full model (as reported)' : ({}, {}),
    '+ MMD balancing'          : ({'lambda_ipm': 0.05}, {}),
    '+ adversarial (GAN)'      : ({'lambda_adv': 0.3},
                                  {'use_adversarial': True, 'lambda_adv': 0.3}),
    '- survival gate'          : ({'lambda_gate': 0.0}, {}),
    '- DR pseudo-outcome loss' : ({'lambda_dr': 0.0},
                                  {'lambda_dr': 0.0, 'lambda_joint': 0.0}),
}

_probe = {}
for _n, (_lw, _ov) in ABLATIONS.items():
    _lp = dict(_LOSS_BASE); _lp.update(_lw)
    _L = CensoringAwareAdversarialLoss(**_lp)
    _c = config.copy(); _c.update(_ov)
    _probe[_n] = (float(_L.lambda_ipm), float(_L.lambda_gate),
                  float(_L.lambda_adv), float(_L.lambda_dr),
                  float(_c.get('lambda_joint', 1.0)),
                  bool(_c.get('use_adversarial', True)))
print('knobs actually reaching the model:')
print(f"  {'variant':<28}{'ipm':>7}{'gate':>7}{'adv':>7}{'dr':>7}{'joint':>7}{'use_adv':>9}")
for _n, _v in _probe.items():
    print(f'  {_n:<28}{_v[0]:>7.3f}{_v[1]:>7.3f}{_v[2]:>7.3f}{_v[3]:>7.3f}'
          f'{_v[4]:>7.3f}{str(_v[5]):>9}')
_bp = _probe['full model (as reported)']
_inert = [n for n, v in _probe.items()
          if n != 'full model (as reported)' and v == _bp]
assert not _inert, f'IDENTICAL to full model, would waste the run: {_inert}'

_chk = _build()
assert _chk.predictor.cov_proj[0].in_features == _chk.n_cov_raw + _N_FUSION_EXTRA, (
    'section 7 _build is NOT the trajectory-augmented builder; run section 7 first')
print(f"  fusion width confirmed: {_chk.predictor.cov_proj[0].in_features} "
      f"(= {_chk.n_cov_raw} time-means + {_N_FUSION_EXTRA} trajectory features)")
del _chk
print('  OK: all variants live, architecture is the reported one.\n')

ABL_FORCE = False

ABL = {}
for _name, (_lw, _over) in ABLATIONS.items():
    _f = f"{OUTPUT_DIR}/tau_oof_abl_{_name.replace(' ','_').replace('+','p').replace('-','m')}.npy"
    if os.path.exists(_f) and not ABL_FORCE:
        ABL[_name] = np.load(_f)
        print(f"[{_name}] loaded cached")
        continue
    print(f"\n{'='*66}\n[{_name}] loss={_lw or 'none'} cfg={_over or 'none'}\n{'='*66}")
    _cfg = config.copy(); _cfg.update(_over)
    _lp = dict(_LOSS_BASE); _lp.update(_lw)
    _loss_v = CensoringAwareAdversarialLoss(**_lp)
    torch.manual_seed(RANDOM_STATE); np.random.seed(RANDOM_STATE)
    _t0 = time.time()
    _r = cross_fitted_ite(_build, train_stage1, train_stage2, _loss_v, _cfg,
                          X_model, W_psm, VFD_psm, D_psm, pad_masks,
                          K=K_FOLDS, seed=RANDOM_STATE,
                          save_dir=f"{OUTPUT_DIR}/_abl_{abs(hash(_name))%10**6}",
                          tau_base=tau_base, verbose=False)
    ABL[_name] = _r['ite_oof']
    np.save(_f, _r['ite_oof'])
    print(f"[{_name}] pooled OOF c = {_r['c_oof']:.4f}   "
          f"({(time.time()-_t0)/60:.1f} min)")

print(f"\n{'='*70}\nABLATION TABLE (pooled out-of-fold, n={len(VFD_psm):,})\n{'='*70}")

_MC_FLOOR = 0.005
_base = c_for_benefit_ci(ABL['full model (as reported)'], VFD_psm, W_psm)['point']
print(f"  {'variant':<28}{'c-for-benefit':>20}{'vs full':>10}{'ITE sd':>9}")
for _n, _v in ABL.items():
    _ci = c_for_benefit_ci(_v, VFD_psm, W_psm)
    _d = _ci['point'] - _base
    print(f"  {_n:<28}{_ci['point']:>7.3f} [{_ci['lower']:.3f},{_ci['upper']:.3f}]"
          f"{_d:>+10.3f}{np.std(_v):>9.2f}")

if 'tau_twin' not in globals():
    tau_twin = np.load(f'{OUTPUT_DIR}/tau_oof_twin.npy')
_ci_rep = c_for_benefit_ci(tau_twin, VFD_psm, W_psm)
print(f"\n  {'reported twin (section 7)':<28}{_ci_rep['point']:>7.3f} "
      f"[{_ci_rep['lower']:.3f},{_ci_rep['upper']:.3f}]"
      f"{_ci_rep['point']-_base:>+10.3f}{np.std(tau_twin):>9.2f}")
print(f"  ^ same configuration, different weight init. |gap| = "
      f"{abs(_ci_rep['point']-_base):.3f} = the initialisation noise floor.")
print(f"  Metric Monte-Carlo noise floor is a further ~{_MC_FLOOR:.3f}.")
print("  Do not claim any ablation effect smaller than the larger of the two.")

print("\n--- paired tests against the full model (same patients) ---")
print("    delta = c(variant) - c(full model). NEGATIVE delta = the variant is")
print("    WORSE, i.e. the removed/added component HELPED. The helper's stock")
print("    message says 'improvement' for either sign; read the verdict line.")
_ABL_STATS = {}
for _n, _v in ABL.items():
    if _n == 'full model (as reported)':
        continue
    print(f"\n{_n}:")
    _t = paired_c_for_benefit_test(_v, ABL['full model (as reported)'],
                                   VFD_psm, W_psm, name_a=_n, name_b='full model')
    _sig = (_t['lower'] > 0) or (_t['upper'] < 0)
    if not _sig:
        _verdict = 'NO MEASURABLE EFFECT (CI includes 0)'
    elif _t['delta'] > 0:
        _verdict = 'variant is BETTER than the reported model'
    else:
        _verdict = 'variant is WORSE => this component genuinely CONTRIBUTES'
    print(f"  VERDICT: {_verdict}")
    if _sig and abs(_t['delta']) < max(_MC_FLOOR, abs(_ci_rep['point'] - _base)):
        print("  CAUTION: significant but SMALLER than the noise floor above; "
              "report it as inconclusive, not as a contribution.")
    _ABL_STATS[_n] = _t

print("\n--- sign stability of each delta across 30 pair-sampling seeds ---")
print(f"  {'variant':<28}{'mean d':>9}{'SD':>8}{'min':>9}{'max':>9}  {'sign stable?':>13}")
_full = ABL['full model (as reported)']
for _n, _v in ABL.items():
    if _n == 'full model (as reported)':
        continue
    _ds = np.array([c_for_benefit(_v, VFD_psm, W_psm, random_state=_s)['c_for_benefit']
                    - c_for_benefit(_full, VFD_psm, W_psm, random_state=_s)['c_for_benefit']
                    for _s in range(30)])
    _stable = bool((_ds > 0).all() or (_ds < 0).all())
    print(f"  {_n:<28}{_ds.mean():>+9.4f}{_ds.std():>8.4f}{_ds.min():>+9.4f}"
          f"{_ds.max():>+9.4f}  {str(_stable):>13}")

